# 08. Cross-Architecture Comparison

This notebook harmonises the public cross-architecture payload and recomputes the layer-resolved `G4` versus `G1` effect-size curves used in the replication figure.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

qwen05 = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
qwen15 = pd.read_csv(DATA / 'qwen15b' / 'pcr_denoised_metrics.csv')
pythia = pd.read_csv(DATA / 'pythia410m' / 'exp16_metrics.csv')
gemma = pd.read_csv(DATA / 'gemma3_1b' / 'exp21a_metrics_full.csv')
invariants = pd.read_csv(DATA / 'analysis' / 'invariant_signatures.csv')

metric_map = {
    'effective_dimension': {
        'qwen05': 'effective_dim',
        'qwen15': 'clean_effective_dimension',
        'pythia': 'effective_dim',
        'gemma': 'effective_dimension',
    },
    'radius_of_gyration': {
        'qwen05': 'radius_of_gyration',
        'qwen15': 'clean_radius_of_gyration',
        'pythia': 'radius_of_gyration',
        'gemma': 'radius_of_gyration',
    },
    'speed': {
        'qwen05': 'speed',
        'qwen15': 'clean_speed',
        'pythia': 'speed',
        'gemma': 'speed',
    },
    'directional_consistency': {
        'qwen05': 'dir_consistency',
        'qwen15': 'clean_dir_consistency',
        'pythia': 'dir_consistency',
        'gemma': 'directional_consistency',
    },
}
        


In [ ]:
def with_groups(frame, correct_col='correct', condition_col='condition'):
    out = frame.copy()
    if 'group' in out.columns:
        return out
    out['group'] = np.select(
        [
            (out[condition_col] == 'direct') & (~out[correct_col].astype(bool)),
            (out[condition_col] == 'direct') & (out[correct_col].astype(bool)),
            (out[condition_col] == 'cot') & (~out[correct_col].astype(bool)),
            (out[condition_col] == 'cot') & (out[correct_col].astype(bool)),
        ],
        ['G1', 'G2', 'G3', 'G4'],
        default='Unknown',
    )
    return out

frames = {
    'Qwen-0.5B': with_groups(qwen05),
    'Qwen-1.5B': with_groups(qwen15),
    'Pythia-410m': with_groups(pythia),
    'Gemma 3 1B': with_groups(gemma),
}
keys = {'Qwen-0.5B': 'qwen05', 'Qwen-1.5B': 'qwen15', 'Pythia-410m': 'pythia', 'Gemma 3 1B': 'gemma'}
styles = {'Qwen-0.5B': ('#3498DB', '-'), 'Qwen-1.5B': ('#E67E22', '--'), 'Pythia-410m': ('#27AE60', ':'), 'Gemma 3 1B': ('#C0392B', '-.')}
rows = []
for model_name, frame in frames.items():
    model_key = keys[model_name]
    max_layer = frame['layer'].max()
    for metric_name, mapping in metric_map.items():
        col = mapping[model_key]
        if col not in frame.columns:
            continue
        for layer, layer_df in frame.groupby('layer'):
            g4 = layer_df[layer_df['group'] == 'G4'][col]
            g1 = layer_df[layer_df['group'] == 'G1'][col]
            rows.append({
                'model': model_name,
                'metric': metric_name,
                'layer': int(layer),
                'rel_depth': float(layer / max_layer) if max_layer else 0.0,
                'cohen_d': cohens_d(g4, g1),
            })
plot_df = pd.DataFrame(rows)
plot_df.head()
        


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.ravel()
for ax, metric_name in zip(axes, metric_map.keys()):
    sub = plot_df[plot_df['metric'] == metric_name]
    for model_name in ['Qwen-0.5B', 'Qwen-1.5B', 'Pythia-410m', 'Gemma 3 1B']:
        model_sub = sub[sub['model'] == model_name]
        color, linestyle = styles[model_name]
        ax.plot(model_sub['rel_depth'], model_sub['cohen_d'], color=color, linestyle=linestyle, label=model_name)
    ax.set_title(metric_name.replace('_', ' ').title())
    ax.set_xlabel('Relative depth')
    ax.set_ylabel("Cohen's d (G4 vs G1)")
axes[0].legend()
plt.tight_layout()
plt.savefig(FIGURES / 'repro_08_cross_architecture.png', dpi=300)
plt.show()

print({'invariant_signature_count': int(len(invariants))})
        
